# Module 1: The New Paradigm — From Yaba to Stockholm
## Lab: Environment Setup & First Protein Visualization

**Lagos Bio-Design Bootcamp** | [JobAiReady](https://github.com/JobAiReady/lagos-bio-design)

---

### Objective
Initialize your cloud-based bio-design environment and visualize your first protein structure.

### Prerequisites
- Google Account (for Colab)
- Basic Python knowledge
- PyMOL 2.5+ installed locally (for Step 3)

### Deliverable
A screenshot of your PyMOL workspace showing the insulin structure colored by confidence.

---
## Step 1: Environment Setup
Clone the repository and install dependencies in this Colab environment.

In [ ]:
# Clone the Lagos Bio-Design Bootcamp repository
!git clone https://github.com/JobAiReady/lagos-bio-design.git
%cd lagos-bio-design

In [ ]:
# Install core dependencies for protein analysis
!pip install -q biopython matplotlib numpy pandas
!pip install -q colabfold[alphafold2] --no-deps 2>/dev/null || echo 'ColabFold optional — see instructions below'

In [ ]:
# Verify installation
import numpy as np
import matplotlib.pyplot as plt
from Bio import SeqIO

print('Environment ready!')
print(f'NumPy: {np.__version__}')

---
## Step 2: AlphaFold Inference — Predict Insulin Structure
We'll use ColabFold to predict the 3D structure of human insulin from its amino acid sequence.

> **Note:** If ColabFold is not available, you can download a pre-computed insulin structure from the [AlphaFold Protein Structure Database](https://alphafold.ebi.ac.uk/entry/P01308).

In [ ]:
# Human Insulin sequence (Chain A + Chain B)
insulin_sequence = "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"

# Write to FASTA file
with open('insulin.fasta', 'w') as f:
    f.write(f'>human_insulin\n{insulin_sequence}\n')

print(f'Sequence length: {len(insulin_sequence)} residues')
print(f'Sequence written to insulin.fasta')

In [ ]:
# Option A: Run ColabFold prediction (requires GPU runtime)
# Uncomment below if ColabFold installed successfully:

# from colabfold.batch import run_batch
# run_batch(input_sequences='insulin.fasta', output_dir='./results')

# Option B: Download pre-computed structure from AlphaFold DB
import urllib.request
import os

os.makedirs('results', exist_ok=True)
url = 'https://alphafold.ebi.ac.uk/files/AF-P01308-F1-model_v4.pdb'
urllib.request.urlretrieve(url, 'results/insulin_predicted.pdb')
print('Downloaded insulin structure from AlphaFold DB')

---
## Step 3: Analyze pLDDT Confidence Scores
Before opening in PyMOL, let's analyze the confidence scores programmatically.

In [ ]:
# Parse the PDB and extract pLDDT scores (stored in B-factor column)
from Bio.PDB import PDBParser
import numpy as np
import matplotlib.pyplot as plt

parser = PDBParser(QUIET=True)
structure = parser.get_structure('insulin', 'results/insulin_predicted.pdb')

# Extract pLDDT (B-factor) per residue
plddt_scores = []
for model in structure:
    for chain in model:
        for residue in chain:
            if residue.get_id()[0] == ' ':  # Skip hetero atoms
                ca = residue['CA'] if 'CA' in residue else list(residue.get_atoms())[0]
                plddt_scores.append(ca.get_bfactor())

plddt = np.array(plddt_scores)
print(f'Residues: {len(plddt)}')
print(f'Mean pLDDT: {plddt.mean():.1f}')
print(f'Min pLDDT:  {plddt.min():.1f}')
print(f'Max pLDDT:  {plddt.max():.1f}')

In [ ]:
# Plot pLDDT per residue
plt.figure(figsize=(12, 4))
plt.bar(range(len(plddt)), plddt, color=[
    '#0053D6' if s > 90 else '#65CBF3' if s > 70 else '#FFDB13' if s > 50 else '#FF7D45'
    for s in plddt
])
plt.axhline(y=90, color='blue', linestyle='--', alpha=0.3, label='Very high (>90)')
plt.axhline(y=70, color='cyan', linestyle='--', alpha=0.3, label='Confident (>70)')
plt.axhline(y=50, color='yellow', linestyle='--', alpha=0.3, label='Low (>50)')
plt.xlabel('Residue Index')
plt.ylabel('pLDDT Score')
plt.title('AlphaFold Confidence (pLDDT) — Human Insulin')
plt.legend()
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

---
## Step 4: PyMOL Visualization (Local)

Download the PDB file and open it in PyMOL on your local machine.

### PyMOL Commands
```
load insulin_predicted.pdb
spectrum b, rainbow_rev, minimum=50, maximum=90
set cartoon_fancy_helices, 1
bg_color white
ray 1200, 900
png insulin_plddt.png
```

**Interpretation:**
- 🔵 Blue regions: High confidence (pLDDT > 90) — well-structured
- 🟡 Yellow regions: Medium confidence — possibly flexible
- 🔴 Red regions: Low confidence — likely disordered

Take a screenshot of your PyMOL visualization for your deliverable!

In [ ]:
# Download the PDB file to your local machine
from google.colab import files
files.download('results/insulin_predicted.pdb')

---
## Summary

In this lab you:
1. Set up a cloud-based Python environment for bio-design
2. Retrieved/predicted a protein structure using AlphaFold
3. Analyzed pLDDT confidence scores programmatically
4. Visualized the structure colored by confidence

**Next:** Module 2 — Building the RFDiffusion → ProteinMPNN pipeline